# 3. Back-translation (English -> Arabic)

Notebook 2 produced the **final English** file for each chapter: Arabic
questionnaires translated, English questionnaires merged in, and the calculated
indicators appended. That file is the fuller one, so the Arabic deliverable is
made by translating it back.

```
COMPENDIUM-ARAB SOCIETY\<Chapter>_EN.xlsx   ->   COMPENDIUM-ARAB SOCIETY\<Chapter>_AR.xlsx
```

This is **not** the same thing as `merged longfiles_AR\<Chapter>_AR.xlsx`. That
one is the original built from the Arabic questionnaires by notebook 1, and this
notebook never touches it. The top-level pair are the deliverables; the
`merged longfiles_*` folders are intermediates.

Run this before notebook 4, so the tabulations have an Arabic file that already
contains the calculated indicators.

## The dictionary only goes one way

`translation dict.xlsx` is Arabic -> English, so this notebook **inverts** it.
Inverting is lossless only where the mapping is one-to-one, and it is not always:
several Arabic spellings can share one English translation - typo variants of the
same survey name, for instance. Where that happens the first Arabic spelling is
used and **every collision is reported**, so the choice is visible rather than
silently made. Any of the spellings is a correct translation.

Anything the dictionary cannot translate stays in English and is collected for
Claude Code to fill in, exactly as in notebook 2.


In [ ]:
"""
CELL: Imports and logging setup.
"""
import difflib
import logging
import re
from collections import defaultdict
from pathlib import Path

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("compendium")


## Config / paths


In [ ]:
"""
CELL: Configuration.
"""
DATA_COLLECTOR_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\DATA COLLECTOR")
TRANSLATION_DICT_PATH = DATA_COLLECTOR_PATH / "translation dict.xlsx"
COMPENDIUM_PATH = Path(r"C:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY")

# Leave CHAPTERS as None to process every chapter found on disk. Set an explicit
# list to restrict one run, e.g. CHAPTERS = ["Poverty"].
CHAPTERS = None

# Columns whose VALUES are never translated - they hold numbers, and a figure
# like 2024 could otherwise collide with a dictionary entry. Their HEADERS are
# still translated, so the Arabic file reads السنة / العدد.
NUMERIC_COLUMNS = {"Year", "Value"}


def discover_chapters():
    """Chapters that notebook 2 produced a final English file for."""
    return sorted(p.name[: -len("_EN.xlsx")]
                  for p in COMPENDIUM_PATH.glob("*_EN.xlsx"))


def chapters_to_process():
    """CHAPTERS if it was set, otherwise whatever notebook 2 produced."""
    if CHAPTERS:
        return list(CHAPTERS)
    found = discover_chapters()
    logger.info(f"Chapters with a final English file: {found}")
    if not found:
        logger.warning("No <Chapter>_EN.xlsx found - run notebook 2 first.")
    return found

# The dictionary lives outside the repo, so git cannot track it. Every update
# also writes this CSV snapshot inside the repo: the .xlsx stays the working
# copy, the CSV is the versioned record - and CSV diffs cleanly where .xlsx
# does not. Commit it after a dictionary change to keep the history.
DICTIONARY_SNAPSHOT_PATH = Path(r"c:\Users\RSHIRINI\OneDrive - United Nations\Desktop\DSS\COMPENDIUM-ARAB SOCIETY\codes\translation_dict_snapshot.csv")


## Inverting the dictionary


In [ ]:
"""
CELL: load_english_to_arabic() - invert the dictionary, and report what that costs.
"""


def load_english_to_arabic():
    """Invert the Arabic -> English dictionary.

    Returns (column_map, value_map, collisions):
        column_map = {English column name: Arabic column name}
        value_map  = {English column name: {English value: Arabic value}}
        collisions = [(column, english, [arabic spellings])] where more than one
                     Arabic value maps to the same English one

    Where several Arabic spellings share an English translation the FIRST is
    used. The collisions are returned rather than swallowed so the run can print
    them - a silent pick would look like certainty it does not have.
    """
    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")

    column_map = {}
    for _, row in dictionary[["col_ar", "col_en"]].dropna().drop_duplicates().iterrows():
        column_map.setdefault(str(row["col_en"]).strip(), str(row["col_ar"]).strip())

    value_map, seen = {}, {}
    for _, row in dictionary.dropna(subset=["col_en", "val_en", "val_ar"]).iterrows():
        english_column = str(row["col_en"]).strip()
        english_value = str(row["val_en"]).strip()
        arabic_value = row["val_ar"]

        bucket = value_map.setdefault(english_column, {})
        key = (english_column, english_value)
        if english_value in bucket:
            seen[key].append(arabic_value)
            continue
        bucket[english_value] = arabic_value
        seen[key] = [arabic_value]

    collisions = [(col, en, spellings) for (col, en), spellings in seen.items()
                  if len(spellings) > 1]
    return column_map, value_map, collisions


ENGLISH_TO_ARABIC_COLUMNS, ENGLISH_TO_ARABIC_VALUES, COLLISIONS = load_english_to_arabic()

logger.info(
    f"Dictionary inverted: {len(ENGLISH_TO_ARABIC_COLUMNS)} column names, "
    f"{sum(len(v) for v in ENGLISH_TO_ARABIC_VALUES.values())} values (English -> Arabic)"
)
if COLLISIONS:
    logger.warning(f"{len(COLLISIONS)} English term(s) have more than one Arabic "
                   f"spelling; the first is used. See the run cell for the list.")


## `translate_to_arabic()` and the gap finder


In [ ]:
"""
CELL: translate_to_arabic() and the gap finder.
"""


def looks_arabic(text):
    """True if the text contains at least one Arabic letter. U+0600-U+06FF is
    the Arabic Unicode block; English text has nothing in it."""
    return any("\u0600" <= character <= "\u06ff" for character in str(text))


def is_web_address(text):
    """A URL or bare domain - a citation that reads the same in either language,
    so not something the dictionary is missing."""
    t = str(text).strip().lower()
    return t.startswith(("http://", "https://", "www.")) or "://" in t


def translate_to_arabic(table):
    """English -> Arabic.

    Values are replaced first and the column renamed second, because the value
    lookup is keyed by the column's ORIGINAL English name - renaming first would
    lose it. For a NUMERIC_COLUMNS column only the header is translated.
    """
    table = table.copy()
    for column in list(table.columns):
        if column in ENGLISH_TO_ARABIC_VALUES and column not in NUMERIC_COLUMNS:
            table[column] = table[column].replace(ENGLISH_TO_ARABIC_VALUES[column])
        if column in ENGLISH_TO_ARABIC_COLUMNS:
            table = table.rename(columns={column: ENGLISH_TO_ARABIC_COLUMNS[column]})
    return table


def find_untranslated(english_table, arabic_table):
    """English text that came through unchanged - the dictionary had no entry.

    Something already in Arabic is not a gap, and neither is a number or a web
    address; asking for those to be "translated" would be noise.
    """
    gaps = []
    for english_column, arabic_column in zip(english_table.columns, arabic_table.columns):
        if english_column in NUMERIC_COLUMNS:
            continue
        pairs = pd.DataFrame({"val_en": english_table[english_column],
                              "val_ar": arabic_table[arabic_column]}).dropna()

        for (english_value, arabic_value), count in pairs.groupby(["val_en", "val_ar"]).size().items():
            if str(english_value).strip() != str(arabic_value).strip():
                continue                                  # translated fine
            if looks_arabic(english_value):
                continue                                  # already Arabic
            if not re.search(r"[A-Za-z]", str(english_value)):
                continue                                  # a number or code
            if is_web_address(english_value):
                continue                                  # a link
            gaps.append({"col_en": english_column, "col_ar": arabic_column,
                         "val_en": english_value, "val_ar": None, "rows": count})
    return gaps


## Filling the gaps

Same loop as notebook 2, in the other direction: `export_untranslated()` ->
Claude Code fills in `val_ar` -> `update_dictionary()` writes it back.


In [ ]:
"""
CELL: export_untranslated() and update_dictionary() - the gap-filling loop.
"""


def export_untranslated(reports, file_name="untranslated_to_arabic.xlsx"):
    """Writes every gap found during the run to one Excel file, shaped like
    translation dict.xlsx so a filled-in row can go straight back into it."""
    rows = [gap for report in reports for gap in report["untranslated"]]
    if not rows:
        logger.info("Nothing untranslated - the dictionary covered every value.")
        return pd.DataFrame()

    gaps = (pd.DataFrame(rows)
            .drop_duplicates(subset=["col_en", "val_en"])
            .sort_values(["col_en", "val_en"])
            .reset_index(drop=True))
    path = COMPENDIUM_PATH / file_name
    gaps.to_excel(path, index=False, engine="openpyxl")
    logger.info(f"Saved {path.name}: {len(gaps):,} value(s) with no dictionary entry. "
                f"Fill in val_ar, then call update_dictionary().")
    return gaps


def update_dictionary(filled, backup=True):
    """Appends reviewed translations to translation dict.xlsx.

    Rows missing either side are skipped, and an (Arabic column, Arabic value)
    pair already present is left alone - so running this twice changes nothing
    the second time. A timestamped backup is written first, because this edits
    the project's source of truth.
    """
    if not isinstance(filled, pd.DataFrame):
        filled = pd.read_excel(filled, engine="openpyxl")

    needed = ["col_ar", "val_ar", "col_en", "val_en"]
    missing = [c for c in needed if c not in filled.columns]
    if missing:
        raise ValueError(f"missing column(s) {missing}; expected {needed}")

    new_rows = filled[needed].dropna()
    new_rows = new_rows[(new_rows["val_ar"].astype(str).str.strip() != "")
                        & (new_rows["val_en"].astype(str).str.strip() != "")]
    if new_rows.empty:
        logger.warning("No completed rows to add - is val_ar filled in?")
        return None

    dictionary = pd.read_excel(TRANSLATION_DICT_PATH, engine="openpyxl")
    already = set(zip(dictionary["col_ar"], dictionary["val_ar"]))
    to_add = new_rows[~new_rows.apply(
        lambda r: (r["col_ar"], r["val_ar"]) in already, axis=1)]
    if to_add.empty:
        logger.info("Every row is already in the dictionary - nothing to add.")
        return dictionary

    if backup:
        stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
        backup_path = TRANSLATION_DICT_PATH.with_name(
            f"{TRANSLATION_DICT_PATH.stem} backup {stamp}.xlsx")
        dictionary.to_excel(backup_path, index=False, engine="openpyxl")
        logger.info(f"Backed up the dictionary to {backup_path.name}")

    updated = pd.concat([dictionary, to_add.reindex(columns=dictionary.columns)],
                        ignore_index=True)
    updated.to_excel(TRANSLATION_DICT_PATH, index=False, engine="openpyxl")
    # utf-8-sig so the Arabic opens correctly if the snapshot is viewed in Excel.
    updated.to_csv(DICTIONARY_SNAPSHOT_PATH, index=False, encoding="utf-8-sig")
    logger.info(f"Snapshot written to {DICTIONARY_SNAPSHOT_PATH.name} - commit it "
                f"to keep the dictionary's history")
    logger.info(f"Added {len(to_add):,} row(s) to {TRANSLATION_DICT_PATH.name} "
                f"({len(dictionary):,} -> {len(updated):,}). Re-run to use them.")
    return updated


## Run - back-translate every chapter


In [ ]:
"""
CELL: Main run - back-translate every chapter's final English file.
"""
print(f"Translating EN -> AR")
print(f"  reading {COMPENDIUM_PATH}\\<Chapter>_EN.xlsx")
print(f"  writing {COMPENDIUM_PATH}\\<Chapter>_AR.xlsx\n")

if COLLISIONS:
    print(f"{len(COLLISIONS)} English term(s) have MORE THAN ONE Arabic spelling. "
          f"The first is used:")
    for column, english, spellings in COLLISIONS[:8]:
        print(f"   [{column}] {english[:60]}")
        print(f"       using {str(spellings[0])[:60]!r}")
    if len(COLLISIONS) > 8:
        print(f"   ... and {len(COLLISIONS) - 8} more")
    print()

REPORTS = []
chapters = chapters_to_process()
total_chapters = len(chapters)
for i, chapter in enumerate(chapters, start=1):
    bar = "#" * i + "-" * (total_chapters - i)
    print(f"[{bar}] chapter {i}/{total_chapters}: {chapter}")

    english_path = COMPENDIUM_PATH / f"{chapter}_EN.xlsx"
    if not english_path.exists():
        logger.info(f"  {chapter}: no {english_path.name} - run notebook 2 first")
        continue

    english_table = pd.read_excel(english_path, engine="openpyxl")
    arabic_table = translate_to_arabic(english_table)

    arabic_path = COMPENDIUM_PATH / f"{chapter}_AR.xlsx"
    arabic_table.to_excel(arabic_path, index=False, engine="openpyxl")

    gaps = find_untranslated(english_table, arabic_table)
    REPORTS.append({"chapter": chapter, "rows": len(arabic_table), "untranslated": gaps})
    logger.info(f"  {chapter}: {english_path.name} -> {arabic_path.name} "
                f"({len(arabic_table):,} rows, {len(gaps)} gap(s))")

print("\n" + "=" * 70)
gap_count = len({(g["col_en"], g["val_en"]) for r in REPORTS for g in r["untranslated"]})
if not REPORTS:
    print("Nothing to translate - run notebook 2 first.")
elif gap_count == 0:
    print("Every value was translated - the dictionary covered all of them.")
else:
    print(f"{gap_count} distinct value(s) had NO dictionary entry and stayed in English:")
    shown = set()
    for report in REPORTS:
        for gap in report["untranslated"]:
            key = (gap["col_en"], gap["val_en"])
            if key in shown:
                continue
            shown.add(key)
            if len(shown) <= 12:
                print(f"   [{gap['col_en']}] {str(gap['val_en'])[:70]}")
    if gap_count > 12:
        print(f"   ... and {gap_count - 12} more")
    print("\nRun export_untranslated(REPORTS), ask Claude Code to fill in val_ar,")
    print("then update_dictionary() - and run this cell again.")


## Gaps


In [ ]:
"""
CELL: Write the gap file for the run above.
"""
UNTRANSLATED = export_untranslated(REPORTS)
UNTRANSLATED.head(20)
